# Do the historical findings recur in the independent Labrador cohort?

The main question is whether the predefined regions from the 2013TSA Labrador study also show evidence of association with behavioral elimination in the independent 2024TSA Labradors. This chapter concerns targeted replication, rather than selecting new targets from the current dogs' results.

We start with the existing inventory of 18 historical markers. Twelve retained measured genotypes after QC and reference alignment; six were excluded. The primary analysis will use the original measured calls at the retained targets, with any missing calls left missing. We do not replace excluded targets with imputed genotypes simply to increase the number tested.

**Current progress:** both GMMAT and GEMMA have been run at the 12 retained measured targets in both Labrador comparisons. This chapter presents their results and an initial check of model behavior. Historical effect matching and further small-sample assessment remain before a directional replication conclusion.

## The two Labrador comparisons

| Comparison | Controls | Behavioral cases | Dogs |
| --- | ---: | ---: | ---: |
| Scheme A: primary | 39 | 7 | 46 |
| Scheme B: sensitivity | 41 | 7 | 48 |

Scheme A uses Validated or Certified dogs as controls. Scheme B adds dogs eliminated for the accepted nonbehavioral reasons. The same seven cases appear in both comparisons. These comparisons will be interpreted separately, with Scheme A providing the primary replication evidence.

In Scheme A, the seven cases comprise three males and four females; controls comprise 27 males and 12 females. Both models adjust for sex, which was specified before inspecting association results. The small case group limits the number of adjustment variables we can reasonably estimate.

## Load and verify the measured targets

The preparation script reads the allele-aligned PLINK files, extracts the retained historical targets, and compares every extracted genotype with an independent PLINK allele-count export. It creates a private analysis matrix and count summaries without performing association tests. The source genotypes remain unchanged.

In [ ]:
from pathlib import Path
import sys

study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
from notebook_support import read_json, run_script, file_digest

replication_folder = study_root / "data/replication/target-preparation-20261004"
manifest_file = replication_folder / "manifest.private.json"
if not manifest_file.exists():
    run_script(study_root, "prepare_replication.py", "--output", replication_folder)
preparation = read_json(manifest_file)
assert preparation["observed_dosages_independently_verified_with_plink"]
assert preparation["source_alignment_manifest_sha256"] == file_digest(
    study_root / "data/aligned/manifest.private.json")
print("Measured historical targets:", preparation["measured_targets"])
print("Historical targets excluded during QC:", preparation["excluded_targets"])

## What allele counts tell us

An allele frequency is the fraction of observed chromosome copies carrying an allele. For these autosomal markers, each dog with a measured genotype contributes two copies. We count cases and controls separately within the current cohort.

The alternate allele is a reference-assembly label, not a declaration that an allele increases risk. A difference in allele frequency is descriptive: it does not account for relatedness, adjust for sex or ancestry, or establish replication. We do not treat the chromosome copies from related dogs as independent observations.

In [ ]:
import csv
from IPython.display import Markdown, display

with (replication_folder / "allele_counts.private.tsv").open(encoding="utf-8", newline="") as handle:
    allele_counts = list(csv.DictReader(handle, delimiter="\t"))
primary = [row for row in allele_counts if row["scenario"] == "success-vs-behelim"]
assert len(primary) == 12
lines = ["| Historical target | Counted allele | Case allele frequency | Control allele frequency |",
         "| --- | --- | ---: | ---: |"]
for row in primary:
    lines.append(f"| {row['historical_target']} | {row['alternate_allele']} | "
                 f"{float(row['case_alt_frequency']):.3f} | {float(row['control_alt_frequency']):.3f} |")
display(Markdown("\n".join(lines)))

### How missing genotypes are handled

This function is the counting step used by the preparation script. A value of 0, 1, or 2 is the number of alternate-allele copies in a measured genotype; `None` denotes a missing measurement. Missing genotypes contribute neither allele copies nor denominator counts.

In [ ]:
def allele_counts_from_genotypes(dosages):
    called = [value for value in dosages if value is not None]
    alternate = sum(called)
    return {"called_dogs": len(called),
            "missing_dogs": len(dosages) - len(called),
            "alt_copies": alternate,
            "total_alleles": 2 * len(called),
            "alt_frequency": alternate / (2 * len(called)) if called else None}

for row in primary:
    assert int(row["case_total_alleles"]) == 2 * int(row["case_called_dogs"])
    assert int(row["control_total_alleles"]) == 2 * int(row["control_called_dogs"])
print("Observed allele denominators checked.")

## Account for related dogs in both models

Both methods ask whether carrying an additional copy of the counted allele is associated with behavioral elimination, after accounting for sex and genetic relationships. GMMAT models a binary outcome directly. GEMMA uses a linear approximation to the same outcome. They use identical measured genotypes and dogs at each target.

The relationship matrix summarizes similarity across thousands of measured background markers. We exclude the chromosome being tested so that the target region does not also contribute to its own background adjustment. We use updated chromosome locations from reference alignment and omit background markers without retained aligned locations.

Only sex is included as a covariate. The 20 components used to draw UMAP and t-SNE are not imported into this small association model. The relationship matrix supplies the broader genetic adjustment, but seven cases still limit the precision and reliability of the tests.

In [ ]:
import numpy as np
from replication_models import standardized_background

background_prefix = study_root / "data/structure/20261002-pca-umap-tsne/LR/02_pruned_genotypes"
aligned_locations = study_root / "data/aligned/not-behelim-vs-behelim/LR/2024TSA_LR.bim"
background_ids, background_chromosomes, standardized_genotypes = standardized_background(
    background_prefix, aligned_locations)

def chromosome_excluded_relationship(genotypes, chromosomes, tested_chromosome, dog_indices):
    """Average standardized genetic similarity outside the tested chromosome."""
    background = genotypes[chromosomes != tested_chromosome][:, dog_indices]
    return background.T @ background / len(background)

print("Aligned background markers:", len(background_chromosomes))

## Run the paired comparison

The two tools live in a separate WSL association environment, so the notebook's existing Python environment does not need to change. For a first installation, run `bash scripts/setup_association_wsl.sh` from the repository's WSL terminal. This installs GEMMA 0.98.5 and GMMAT 1.5.0 and saves the installed package information.

The following cell calls the paired analysis script when completed results are absent. A fresh run requires a WSL/Linux notebook kernel. A completed run can be reviewed from either platform. Use a new output folder when changing the analysis settings; the script checks its saved specification before resuming.

Each target uses measured genotypes only. A missing control genotype at one target reduces that comparison to 45 dogs in Scheme A and 47 in Scheme B. All seven cases have measured genotypes at every retained target.

In [ ]:
model_folder = study_root / "data/replication/models-aligned-20261004"
model_manifest_path = model_folder / "manifest.private.json"
if not model_manifest_path.exists():
    run_script(study_root, "replication_models.py", "--output", model_folder, linux=True)
models = read_json(model_manifest_path)
assert models["status"] == "paired_models_complete"
assert len(models["jobs"]) == 24

from notebook_support import study_path
saved_plan = read_json(model_folder / "plan.private.json")
assert saved_plan["config"] == read_json(study_root / "config/replication.json")
assert saved_plan["script_sha256"] == file_digest(study_root / "scripts/replication_models.py")
assert saved_plan["R_script_sha256"] == file_digest(study_root / "scripts/replication_models.R")
for source, digest in saved_plan["sources"].items():
    assert file_digest(study_path(study_root, source)) == digest
for job in models["jobs"]:
    result_path = study_path(study_root, job["result"])
    assert file_digest(result_path) == read_json(
        result_path.parent / "result-receipt.private.json")["sha256"]
print("Both methods completed at 12 targets in each comparison; source and result checks passed.")

### Check one relationship matrix independently

This example reconstructs the first target's matrix from the background genotypes and that target's dog order. It checks that the calculation shown above matches the matrix supplied to both tools.

In [ ]:
example_result = study_path(study_root, models["jobs"][0]["result"])
example_folder = example_result.parent
example = read_json(example_folder / "input.private.json")
with (example_folder / "dogs.private.tsv").open(encoding="utf-8", newline="") as handle:
    example_dogs = list(csv.DictReader(handle, delimiter="\t"))
positions = {dog_id: i for i, dog_id in enumerate(background_ids)}
indices = [positions[dog["id"]] for dog in example_dogs]
tested_chromosome = int(example["canFam4_chromosome"].removeprefix("chr"))
reconstructed = chromosome_excluded_relationship(
    standardized_genotypes, background_chromosomes, tested_chromosome, indices)
supplied = np.loadtxt(example_folder / "kinship.txt")
assert np.allclose(reconstructed, supplied, atol=1e-12)
assert np.linalg.eigvalsh(supplied).min() > -1e-8
print("Reconstructed matrix matches both models' input for", example["marker"])

## Compare the association results

We compare score-test p-values from both methods. Neither method is selected because it produces a smaller p-value. The same predefined targets are shown for Scheme A and Scheme B; Scheme B is a sensitivity comparison using the same seven cases.

The nominal 0.05 line is descriptive. If either method can provide a positive finding, a conservative comparison uses the original 18 historical targets and both methods: `0.05 / (18 × 2)`, approximately 0.00139. Six excluded historical targets remain part of that predefined family. Crossing this line alone would still require reliable small-sample behavior and a matched historical effect before declaring replication.

In [ ]:
figure_folder = study_root / "results/private/version2/06_2013tsa_lr_replication"
run_script(study_root, "summarize_replication.py", "--run-dir", model_folder,
           "--results-dir", figure_folder)
summary = read_json(figure_folder / "summary.private.json")
from IPython.display import Image
display(Image(filename=str(figure_folder / "paired_score_tests.png")))

In [ ]:
with (figure_folder / "paired_models.private.tsv").open(encoding="utf-8", newline="") as handle:
    paired_results = list(csv.DictReader(handle, delimiter="\t"))
lines = ["| Historical target | GMMAT score p | GEMMA score p | Dogs | GMMAT effect fit |",
         "| --- | ---: | ---: | ---: | --- |"]
primary_results = [r for r in paired_results if r["scenario"] == "success-vs-behelim"]
for i in range(0, len(primary_results), 2):
    logistic, linear = primary_results[i:i+2]
    fit = "Converged" if logistic["effect_fit_converged"] == "True" else "Unstable; effect omitted"
    lines.append(f"| {logistic['historical_target']} | {float(logistic['score_p']):.4g} | "
                 f"{float(linear['score_p']):.4g} | {logistic['dogs']} | {fit} |")
display(Markdown("\n".join(lines)))

### What does this comparison show?

None of the 12 measured targets had a score-test p-value below 0.05 with either method in either comparison. The smallest p-values in Scheme A were 0.130 for GMMAT and 0.231 for GEMMA; in Scheme B they were 0.105 and 0.237. There is no clear association signal in these comparisons. With only seven cases, this does not establish that the historical findings were false.

All 24 observed GMMAT null models converged. Its effect estimates did not converge at three targets in either cohort: the retained targets on chromosome 4, chromosome 13, and the second chromosome 36 target. GEMMA returned finite test results, but its alternative-model variance ratio reached the upper search limit in every comparison. These fit limitations remain part of the interpretation.

### Interpret the effect estimates carefully

GMMAT's alternative-model coefficient is a change in log odds per additional counted allele; exponentiating it gives an odds ratio. GEMMA's coefficient is a change on the 0/1 outcome scale and is not an odds ratio. The alternate-allele label does not identify the historical risk allele.

Sparse genotypes can make a logistic effect estimate unstable even when the null model and its score test run successfully. The results distinguish those fits. A very large coefficient or standard error is not evidence of a very large biological effect; a nonconverged alternative fit is omitted from effect interpretation. Complete numerical estimates, fit flags and warnings are retained in the private result files.

## How do the tests behave with seven cases?

For each target and comparison, we generated 100 simulated datasets under the fitted GMMAT model without a target effect. These retain the measured genotypes, sex, genetic relationships and exactly seven cases. Both score tests are run on each simulated dataset after refitting the model.

The figure shows how often p-values fell below 0.05 in successful paired fits, with 95% intervals. A well-behaved nominal test would reject about 5% of these datasets under this particular null model. Failed fits are counted separately in the table, because removing difficult datasets can affect the apparent behavior.

This is a preliminary check under one estimated logistic model. It does not establish reliable probabilities near 0.00139, test every possible source of confounding, or validate the linear model under an independent simulation model. The seven-case limitation remains.

In [ ]:
display(Image(filename=str(figure_folder / "null_screen.png")))
lines = ["| Target, Scheme A | Valid paired draws | Failed draws | GMMAT p < 0.05 | GEMMA p < 0.05 |",
         "| --- | ---: | ---: | ---: | ---: |"]
for i in range(0, len(primary_results), 2):
    logistic, linear = primary_results[i:i+2]
    lines.append(f"| {logistic['historical_target']} | {logistic['null_replicates_valid']} | "
                 f"{logistic['null_replicates_failed']} | {float(logistic['null_rate_below_05']):.1%} | "
                 f"{float(linear['null_rate_below_05']):.1%} |")
display(Markdown("\n".join(lines)))
print("All comparisons:", summary["paired_null_draws_valid"], "valid paired draws and",
      summary["paired_null_draws_failed"], "failed draws.")

Across all 24 target-and-cohort comparisons, 2,360 of 2,400 simulated datasets produced valid paired tests; 40 had failed fits. The intervals are broad, and the apparent rejection rates vary across targets. This preliminary screen does not support declaring either method fully calibrated for this small cohort.

## What remains before a replication conclusion?

The model comparison is complete, but a directional replication conclusion also needs matched historical effect information. The original publication describes some genotype-specific effects; those are not automatically equivalent to a per-allele effect. The existing 18-target inventory is retained while those allele and contrast definitions are reconciled.

Further small-sample assessment is needed before interpreting an unusually small p-value as strong evidence. Conversely, a nonsignificant result in seven cases does not establish that a historical finding is false. The two methods and the sensitivity cohort provide views of the same dogs, rather than independent replication studies.

Source: [2013TSA cohort publication](https://www.nature.com/articles/s41598-022-18698-4). Supporting implementation details are in the [targeted replication methods note](../docs/methods/06%20Targeted%20Labrador%20replication.md).